<a href="https://colab.research.google.com/github/viratojha22/benzothiazinone-vina-docking/blob/main/Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina
!./vina --version

AutoDock Vina v1.2.5


In [ ]:
!pip install rdkit

In [ ]:
!pip install meeko

In [ ]:
!pip install py3Dmol

In [ ]:
!pip install openbabel-wheel

In [ ]:
from rdkit import Chem
import meeko, py3Dmol
try:
    from openbabel import openbabel
except ImportError:
    print("openbabel not available (optional)")
print("all good")

all good


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from google.colab import files
uploaded = files.upload()   # select the receptor PDB and all ligand files together
print(list(uploaded.keys()))

Saving Preparedsecond6hez.pdb to Preparedsecond6hez.pdb
['Preparedsecond6hez.pdb']


In [ ]:
!pip install -q chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
df = pd.read_csv("DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv", sep=";")   # ChEMBL CSVs are often semicolon-separated
ids = df["Molecule ChEMBL ID"].dropna().unique().tolist()
print(len(ids), ids[:5])

FileNotFoundError: [Errno 2] No such file or directory: 'DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv'

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose the downloaded ChEMBL CSV
print(list(uploaded.keys()))

Saving DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv to DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv
['DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv']


In [ ]:
import os
work = "/content/docking_project"
for sub in ["", "ligands", "results", "logs"]:
    os.makedirs(os.path.join(work, sub), exist_ok=True)
print(os.listdir(work))

['logs', 'ligands', 'results']


In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

core_smiles = "O=C1N=CSc2ccccc12"   # 4H-1,3-benzothiazin-4-one

res = new_client.substructure.filter(smiles=core_smiles).only(
    ["molecule_chembl_id", "pref_name", "molecule_structures"]
)

rows = []
for m in res:
    s = m.get("molecule_structures")
    if s and s.get("canonical_smiles"):
        rows.append({"molecule_chembl_id": m["molecule_chembl_id"],
                     "name": m.get("pref_name"),
                     "smiles": s["canonical_smiles"]})

df = pd.DataFrame(rows)
print(len(df))
df.head(10)

/usr/local/lib/python3.13/dist-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. U

708


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,molecule_chembl_id,name,smiles
0,CHEMBL1223570,BRETSCHNEIDERAZINE A,COc1cc2sc(=S)n(C)c(=O)c2cc1O
1,CHEMBL1223571,BRETSCHNEIDERAZINE B,COc1ccc2c(=O)n(C)c(=S)sc2c1OC
2,CHEMBL1303553,None,O=C(Cc1nc(=O)c2ccccc2s1)Nc1ccccc1
3,CHEMBL1336615,None,O=c1nc(Nc2ccc(Br)cc2)sc2ccc([N+](=O)[O-])cc12
4,CHEMBL1338502,None,O=C(Nc1nc(=O)c2ccccc2s1)c1ccccc1
5,CHEMBL1341904,None,COCCn1c(=O)sc2ccccc2c1=O
6,CHEMBL1387491,None,COc1ccc(Nc2nc(=O)c3cc([N+](=O)[O-])ccc3s2)cc1
7,CHEMBL1388972,None,CCN(CC)C(=S)SCC(=O)Nc1nc(=O)c2ccccc2s1
8,CHEMBL1410288,None,CC(=O)Nc1nc(=O)c2ccccc2s1
9,CHEMBL1439614,None,O=c1sc2ccccc2c(=O)n1Cc1ccccc1F


In [ ]:
from google.colab import files

df.to_csv("/content/docking_project/chembl_benzothiazinone_hits_raw.csv", index=False)
files.download("/content/docking_project/chembl_benzothiazinone_hits_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/docking_project/chembl_benzothiazinones.csv")

FileNotFoundError: Cannot find file: /content/docking_project/chembl_benzothiazinones.csv

In [ ]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
import pandas as pd, os

work = "/content/docking_project"
os.makedirs(work, exist_ok=True)

chooser = rdMolStandardize.LargestFragmentChooser()
core = Chem.MolFromSmarts("[#8]=[#6]1~[#7]~[#6]~[#16]~c2ccccc12")

rows = []
for _, r in df.iterrows():
    m = Chem.MolFromSmiles(r["smiles"])
    if m is None:
        continue
    m = chooser.choose(m)
    if m.HasSubstructMatch(core):
        r = r.copy()
        r["smiles"] = Chem.MolToSmiles(m)
        rows.append(r)

clean = pd.DataFrame(rows).drop_duplicates("smiles").reset_index(drop=True)
clean.to_csv(f"{work}/chembl_benzothiazinones.csv", index=False)

print("before:", len(df), "after:", len(clean))
print(os.listdir(work))

before: 708 after: 706
['chembl_benzothiazinones.csv', 'logs', 'ligands', 'chembl_benzothiazinone_hits_raw.csv', 'results']


In [ ]:
from google.colab import files
files.download("/content/docking_project/chembl_benzothiazinones.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
import pandas as pd

clean = pd.read_csv("/content/docking_project/chembl_benzothiazinones.csv")

def props(smi):
    m = Chem.MolFromSmiles(smi)
    return pd.Series({
        "MW": Descriptors.MolWt(m),
        "logP": Crippen.MolLogP(m),
        "HBD": rdMolDescriptors.CalcNumHBD(m),
        "HBA": rdMolDescriptors.CalcNumHBA(m),
        "RotB": rdMolDescriptors.CalcNumRotatableBonds(m),
    })

clean = pd.concat([clean, clean["smiles"].apply(props)], axis=1)
nitro = Chem.MolFromSmarts("[N+](=O)[O-]")
clean["has_nitro"] = clean["smiles"].apply(lambda s: Chem.MolFromSmiles(s).HasSubstructMatch(nitro))

clean.to_csv("/content/docking_project/chembl_benzothiazinones_props.csv", index=False)
print(clean[["MW", "logP", "HBD", "HBA", "RotB"]].describe().round(1))
print("with nitro group:", clean["has_nitro"].sum())

          MW   logP    HBD    HBA   RotB
count  706.0  706.0  706.0  706.0  706.0
mean   477.1    3.9    0.3    7.8    4.4
std     78.0    1.1    0.5    1.6    1.7
min    178.2    0.4    0.0    3.0    0.0
25%    444.4    3.1    0.0    7.0    3.0
50%    483.3    3.9    0.0    8.0    4.0
75%    518.4    4.6    1.0    9.0    5.0
max    810.8    7.4    2.0   13.0   11.0
with nitro group: 610


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.SimDivFilters import rdSimDivPickers
import pandas as pd

clean = pd.read_csv("/content/docking_project/chembl_benzothiazinones_props.csv")

sub = clean[(clean["has_nitro"]) & (clean["MW"].between(400, 520))].reset_index(drop=True)
print("candidates:", len(sub))

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in sub["smiles"]]

n = 100   # number of ligands
picker = rdSimDivPickers.MaxMinPicker()
idx = list(picker.LazyBitVectorPick(fps, len(fps), min(n, len(fps)), seed=42))

sel = sub.iloc[idx].reset_index(drop=True)
sel.to_csv("/content/docking_project/selected_ligands.csv", index=False)
print(len(sel), "selected")
sel[["molecule_chembl_id", "MW", "logP", "smiles"]].head(10)

candidates: 394


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


100 selected


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,molecule_chembl_id,MW,logP,smiles
0,CHEMBL4853819,488.572,3.72810,CN(Cc1csc(-c2ccccc2)n1)c1nc(=O)c2cc(S(C)(=O)=O...
1,CHEMBL5542887,459.446,3.93214,Cc1c(C(F)(F)F)c(C)c2c(=O)nc(N3CCC4(CC3)OC[C@H]...
2,CHEMBL3884595,461.956,3.73710,O=c1nc(N2CCN(C(=S)Nc3ccc([N+](=O)[O-])cc3)CC2)...
3,CHEMBL4461232,513.542,4.28660,CON=C1CCC(CN2CCN(c3nc(=O)c4cc(C(F)(F)F)cc([N+]...
4,CHEMBL5193230,499.917,4.19750,O=c1ccc2cc(-n3cc(CSc4nc(=O)c5cc(Cl)cc([N+](=O)...
5,CHEMBL5203558,421.360,4.22770,CN(c1ccc2[nH]ncc2c1)c1nc(=O)c2cc(C(F)(F)F)cc([...
6,CHEMBL5069638,492.623,3.20310,CS(=O)(=O)c1cc([N+](=O)[O-])c2sc(N3CCN(CC4CCCC...
7,CHEMBL2417567,437.434,2.53000,CC(C)(C)OC(=O)N1CCN(c2nc(=O)c3cc([N+](=O)[O-])...
8,CHEMBL5179919,516.525,4.97760,CC(C)n1c2c([n+](C)c1-c1ccccc1)CN(c1nc(=O)c3cc(...
9,CHEMBL2177265,439.371,3.96680,O=c1nc(NCc2ccc3c(c2)OCCO3)sc2c([N+](=O)[O-])cc...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy
import pandas as pd, os

sel = pd.read_csv("/content/docking_project/selected_ligands.csv")
out_dir = "/content/docking_project/ligands"
os.makedirs(out_dir, exist_ok=True)

preparator = MoleculePreparation()
ok, failed = [], []

for _, r in sel.iterrows():
    cid = r["molecule_chembl_id"]
    try:
        mol = Chem.AddHs(Chem.MolFromSmiles(r["smiles"]))
        if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
            raise ValueError("3D embedding failed")
        AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
        setups = preparator.prepare(mol)
        pdbqt, is_ok, err = PDBQTWriterLegacy.write_string(setups[0])
        if not is_ok:
            raise ValueError(err)
        with open(f"{out_dir}/{cid}.pdbqt", "w") as f:
            f.write(pdbqt)
        ok.append(cid)
    except Exception as e:
        failed.append((cid, str(e)[:60]))

print("prepared:", len(ok),

_IncompleteInputError: incomplete input (235191152.py, line 30)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy
import pandas as pd, os

sel = pd.read_csv("/content/docking_project/selected_ligands.csv")
out_dir = "/content/docking_project/ligands"
os.makedirs(out_dir, exist_ok=True)

preparator = MoleculePreparation()
ok, failed = [], []

for _, r in sel.iterrows():
    cid = r["molecule_chembl_id"]
    try:
        mol = Chem.AddHs(Chem.MolFromSmiles(r["smiles"]))
        if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
            raise ValueError("3D embedding failed")
        AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
        setups = preparator.prepare(mol)
        pdbqt, is_ok, err = PDBQTWriterLegacy.write_string(setups[0])
        if not is_ok:
            raise ValueError(err)
        with open(f"{out_dir}/{cid}.pdbqt", "w") as f:
            f.write(pdbqt)
        ok.append(cid)
    except Exception as e:
        failed.append((cid, str(e)[:60]))

print("prepared:", len(ok))
print("failed:", len(failed))
print(failed[:5])

prepared: 100
failed: 0
[]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import os
files_ = [f for f in os.listdir("/content/docking_project/ligands") if f.endswith(".pdbqt")]
print(len(files_), "PDBQT files")
print(sorted(files_)[:5])

100 PDBQT files
['CHEMBL2177265.pdbqt', 'CHEMBL2177268.pdbqt', 'CHEMBL2177269.pdbqt', 'CHEMBL2177271.pdbqt', 'CHEMBL2177273.pdbqt']


In [ ]:
import os
for root, d, f in os.walk("/content"):
    if "sample_data" in root or ".config" in root:
        continue
    for name in f:
        print(os.path.join(root, name))

/content/DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv
/content/Preparedsecond6hez.pdb
/content/vina
/content/docking_project/chembl_benzothiazinones.csv
/content/docking_project/chembl_benzothiazinones_props.csv
/content/docking_project/selected_ligands.csv
/content/docking_project/chembl_benzothiazinone_hits_raw.csv
/content/docking_project/ligands/CHEMBL4864015.pdbqt
/content/docking_project/ligands/CHEMBL5088001.pdbqt
/content/docking_project/ligands/CHEMBL4866274.pdbqt
/content/docking_project/ligands/CHEMBL4852447.pdbqt
/content/docking_project/ligands/CHEMBL5624960.pdbqt
/content/docking_project/ligands/CHEMBL5171817.pdbqt
/content/docking_project/ligands/CHEMBL4084590.pdbqt
/content/docking_project/ligands/CHEMBL5196214.pdbqt
/content/docking_project/ligands/CHEMBL5171533.pdbqt
/content/docking_project/ligands/CHEMBL5199604.pdbqt
/content/docking_project/ligands/CHEMBL5089268.pdbqt
/content/docking_project/ligands/CHEMBL5403245.pdbqt
/content/docking_project/liga

In [ ]:
import os, numpy as np
from collections import defaultdict

# 1. Files in the session
for root, d, f in os.walk("/content"):
    if "sample_data" in root or ".config" in root:
        continue
    for name in f:
        print(os.path.join(root, name))

# 2. Download 6HEZ if it isn't there, then find the centre of each non-water HETATM group
p = "/content/docking_project/6HEZ.pdb"
if not os.path.exists(p):
    os.system(f"wget -q https://files.rcsb.org/download/6HEZ.pdb -O {p}")

groups = defaultdict(list)
for line in open(p):
    if line.startswith("HETATM") and line[17:20].strip() != "HOH":
        key = (line[17:20].strip(), line[21], line[22:26].strip())
        groups[key].append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

print()
for k, xyz in groups.items():
    c = np.mean(xyz, axis=0).round(1)
    print(k, len(xyz), "atoms, centre:", c)

/content/DOWNLOAD-nxTtU8zSRlvc27e0vQ91gnDRG9TLscpqDI36Oc2DV3M_eq_.csv
/content/Preparedsecond6hez.pdb
/content/vina
/content/docking_project/chembl_benzothiazinones.csv
/content/docking_project/chembl_benzothiazinones_props.csv
/content/docking_project/selected_ligands.csv
/content/docking_project/chembl_benzothiazinone_hits_raw.csv
/content/docking_project/ligands/CHEMBL4864015.pdbqt
/content/docking_project/ligands/CHEMBL5088001.pdbqt
/content/docking_project/ligands/CHEMBL4866274.pdbqt
/content/docking_project/ligands/CHEMBL4852447.pdbqt
/content/docking_project/ligands/CHEMBL5624960.pdbqt
/content/docking_project/ligands/CHEMBL5171817.pdbqt
/content/docking_project/ligands/CHEMBL4084590.pdbqt
/content/docking_project/ligands/CHEMBL5196214.pdbqt
/content/docking_project/ligands/CHEMBL5171533.pdbqt
/content/docking_project/ligands/CHEMBL5199604.pdbqt
/content/docking_project/ligands/CHEMBL5089268.pdbqt
/content/docking_project/ligands/CHEMBL5403245.pdbqt
/content/docking_project/liga

In [ ]:
import numpy as np
from collections import defaultdict, Counter

prep = "/content/Preparedsecond6hez.pdb"
ref  = "/content/docking_project/6HEZ.pdb"

atoms, chains, het = [], Counter(), Counter()
for line in open(prep):
    if line.startswith(("ATOM", "HETATM")):
        if line.startswith("ATOM"):
            atoms.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            chains[line[21]] += 1
        else:
            het[(line[17:20].strip(), line[21])] += 1
atoms = np.array(atoms)
print("protein atoms per chain:", dict(chains))
print("HETATM in prepared file:", dict(het))

sites = defaultdict(list)
for line in open(ref):
    if line.startswith("HETATM") and line[17:20].strip() == "0SK":
        sites[line[21]].append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

for ch, xyz in sites.items():
    xyz = np.array(xyz)
    d = np.linalg.norm(atoms[:, None, :] - xyz[None, :, :], axis=2).min(axis=0)
    center = xyz.mean(axis=0).round(2)
    size = np.maximum(xyz.max(axis=0) - xyz.min(axis=0) + 10, 20).round(1)
    print(f"site chain {ch}: {(d < 4.5).sum()}/{len(xyz)} ligand atoms within 4.5 A of your protein | centre {center} | box {size}")

protein atoms per chain: {'A': 4035, 'B': 3933}
HETATM in prepared file: {}
site chain A: 25/28 ligand atoms within 4.5 A of your protein | centre [ 13.87 -20.62  37.13] | box [20.1 20.  20. ]
site chain B: 24/28 ligand atoms within 4.5 A of your protein | centre [-24.4  -26.33   5.07] | box [20.  23.1 20. ]


In [ ]:
import numpy as np

prep = "/content/Preparedsecond6hez.pdb"
out  = "/content/docking_project/receptor_A.pdb"

kept, has_H, n_prot = [], 0, 0
for line in open(prep):
    rec = line[:6].strip()
    if rec == "ATOM" and line[21] == "A":
        kept.append(line); n_prot += 1
        el = line[76:78].strip() or line[12:16].strip()[0]
        has_H += (el == "H")
    elif rec == "HETATM" and line[21] == "A" and line[17:20].strip() == "FAD":
        kept.append(line)
kept.append("END\n")
open(out, "w").writelines(kept)

print("chain A protein atoms:",

_IncompleteInputError: incomplete input (3738423321.py, line 18)

In [ ]:
prep = "/content/Preparedsecond6hez.pdb"
out = "/content/docking_project/receptor_A.pdb"

kept = []
n_prot = 0
n_H = 0
for line in open(prep):
    rec = line[:6].strip()
    if rec == "ATOM" and line[21] == "A":
        kept.append(line)
        n_prot += 1
        el = line[76:78].strip()
        if el == "H" or line[12:16].strip().startswith("H"):
            n_H += 1
    elif rec == "HETATM" and line[21] == "A":
        if line[17:20].strip() == "FAD":
            kept.append(line)

kept.append("END\n")
open(out, "w").writelines(kept)

print("protein atoms:", n_prot)
print("hydrogens:", n_H)
print("lines written:", len(kept))

protein atoms: 4035
hydrogens: 750
lines written: 4036


In [ ]:
import numpy as np, glob, os

# Box from the chain A BTZ043 (0SK) site in 6HEZ
site = []
for l in open("/content/docking_project/6HEZ.pdb"):
    if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A":
        site.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
site = np.array(site)
c = site.mean(axis=0).round(2)
s = np.maximum(site.max(axis=0) - site.min(axis=0) + 10, 20).round(1)
print("centre:", c, "size:", s)

# First ligand in the folder
lig = sorted(glob.glob("/content/docking_project/ligands/*.pdbqt"))[0]
print("ligand:", lig)

out = "/content/docking_project/results/test_out.pdbqt"
cmd = (f"/content/vina --receptor /content/docking_project/receptor_A.pdbqt "
       f"--ligand {lig} "
       f"--center_x {c[0]} --center_y {c[1]} --center_z {c[2]} "
       f"--size_x {s[0]} --size_y {s[1]} --size_z {s[2]} "
       f"--exhaustiveness 8 --num_modes 5 --out {out}")
!{cmd}

centre: [ 13.87 -20.62  37.13] size: [20.1 20.  20. ]
ligand: /content/docking_project/ligands/CHEMBL2177265.pdbqt
AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#     

In [ ]:
import os
src = "/content/docking_project/receptor_A.pdb"
dst = "/content/docking_project/receptor_A.pdbqt"
print("receptor_A.pdb exists:", os.path.exists(src))
print("receptor_A.pdbqt exists:", os.path.exists(dst))
print(os.popen("which obabel").read() or "obabel not found")

receptor_A.pdb exists: True
receptor_A.pdbqt exists: False
/usr/local/bin/obabel



In [ ]:
src = "/content/docking_project/receptor_A.pdb"
dst = "/content/docking_project/receptor_A.pdbqt"

!obabel {src} -O {dst} -xr -p 7.4

import os
print("exists:", os.path.exists(dst))
!grep -c "^ATOM" {dst}
!grep -c "^HETATM" {dst}

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is /content/docking_project/receptor_A.pdb)

1 molecule converted
exists: True
4089
0


In [ ]:
import os, glob, numpy as np

dst = "/content/docking_project/receptor_A.pdbqt"
print("receptor exists:", os.path.exists(dst))

if os.path.exists(dst):
    site = []
    for l in open("/content/docking_project/6HEZ.pdb"):
        if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A":
            site.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
    site = np.array(site)
    c = site.mean(axis=0).round(2)
    s = np.maximum(site.max(axis=0) - site.min(axis=0) + 10, 20).round(1)

    lig = sorted(glob.glob("/content/docking_project/ligands/*.pdbqt"))[0]
    out = "/content/docking_project/results/test_out.pdbqt"
    cmd = (f"/content/vina --receptor {dst} --ligand {lig} "
           f"--center_x {c[0]} --center_y {c[1]} --center_z {c[2]} "
           f"--size_x {s[0]} --size_y {s[1]} --size_z {s[2]} "
           f"--exhaustiveness 8 --num_modes 5 --out {out}")
    !{cmd}

receptor exists: True
AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/c

In [ ]:
import os, glob, subprocess, numpy as np, pandas as pd

work = "/content/docking_project"
rec = f"{work}/receptor_A.pdbqt"
res_dir = f"{work}/results/poses"
log_dir = f"{work}/logs"
os.makedirs(res_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

site = []
for l in open(f"{work}/6HEZ.pdb"):
    if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A":
        site.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
site = np.array(site)
c = site.mean(axis=0).round(2)
s = np.maximum(site.max(axis=0) - site.min(axis=0) + 10, 20).round(1)

def best_score(path):
    for line in open(path):
        if line.startswith("REMARK VINA RESULT"):
            return float(line.split()[3])
    return None

ligs = sorted(glob.glob(f"{work}/ligands/*.pdbqt"))
print("ligands:", len(ligs))

ligands: 100


In [ ]:
import os, glob, subprocess, numpy as np, pandas as pd

work = "/content/docking_project"
rec = f"{work}/receptor_A.pdbqt"
res_dir = f"{work}/results/poses"
log_dir = f"{work}/logs"
os.makedirs(res_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

site = []
for l in open(f"{work}/6HEZ.pdb"):
    if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A":
        site.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
site = np.array(site)
c = site.mean(axis=0).round(2)
s = np.maximum(site.max(axis=0) - site.min(axis=0) + 10, 20).round(1)

def best_score(path):
    for line in open(path):
        if line.startswith("REMARK VINA RESULT"):
            return float(line.split()[3])
    return None

ligs = sorted(glob.glob(f"{work}/ligands/*.pdbqt"))
print("ligands:", len(ligs))

ligands: 100


In [ ]:
import time
t0 = time.time()

for i, lig in enumerate(ligs, 1):
    cid = os.path.basename(lig).replace(".pdbqt", "")
    out = f"{res_dir}/{cid}_out.pdbqt"
    if os.path.exists(out) and best_score(out) is not None:
        continue
    cmd = [
        "/content/vina", "--receptor", rec, "--ligand", lig,
        "--center_x", str(c[0]), "--center_y", str(c[1]), "--center_z", str(c[2]),
        "--size_x", str(s[0]), "--size_y", str(s[1]), "--size_z", str(s[2]),
        "--exhaustiveness", "8", "--num_modes", "5", "--out", out,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    open(f"{log_dir}/{cid}.log", "w").write(r.stdout + r.stderr)
    print(i, cid, best_score(out), f"{(time.time()-t0)/60:.1f} min")

print("done")

1 CHEMBL2177265 -10.527 1.1 min
2 CHEMBL2177268 -10.053 1.9 min
3 CHEMBL2177269 -9.393 3.7 min
4 CHEMBL2177271 -10.055 5.6 min
5 CHEMBL2177273 -8.435 7.0 min
6 CHEMBL2177275 -9.347 8.7 min
7 CHEMBL2177280 -8.485 9.3 min
8 CHEMBL2417567 -8.598 10.1 min
9 CHEMBL2419037 -10.167 10.5 min
10 CHEMBL3330324 -9.598 13.1 min
11 CHEMBL3394191 -11.355 13.8 min
12 CHEMBL3397351 -8.432 15.9 min
13 CHEMBL3884595 -8.043 16.7 min
14 CHEMBL3885136 -9.146 17.2 min
15 CHEMBL3885427 -9.631 18.5 min
16 CHEMBL4067248 -8.144 20.2 min
17 CHEMBL4084590 -7.736 21.8 min
18 CHEMBL4238874 -8.626 23.3 min
19 CHEMBL4247492 -8.279 24.6 min
20 CHEMBL4435282 -9.788 26.2 min
21 CHEMBL4461232 -8.635 28.6 min
22 CHEMBL4466845 -10.679 29.2 min
23 CHEMBL4467871 -9.185 30.7 min
24 CHEMBL4483310 -9.572 32.8 min
25 CHEMBL4532298 -9.271 34.6 min
26 CHEMBL4536139 -9.04 36.4 min
27 CHEMBL4556916 -9.915 38.1 min
28 CHEMBL4581600 -8.588 39.1 min
29 CHEMBL4776641 -9.238 40.5 min
30 CHEMBL4845692 -9.648 42.4 min
31 CHEMBL4847341 -10.

In [ ]:
import os, glob, shutil
import pandas as pd

work = "/content/docking_project"
res_dir = f"{work}/results/poses"
top_dir = f"{work}/results/top20"

def best_score(path):
    for line in open(path):
        if line.startswith("REMARK VINA RESULT"):
            return float(line.split()[3])
    return None

rows = []
for f in glob.glob(f"{res_dir}/*_out.pdbqt"):
    cid = os.path.basename(f).replace("_out.pdbqt", "")
    sc = best_score(f)
    if sc is not None:
        rows.append({"molecule_chembl_id": cid, "best_affinity_kcal_mol": sc})

scores = pd.DataFrame(rows).sort_values("best_affinity_kcal_mol").reset_index(drop=True)
print("ligands docked so far:", len(scores))

ligands docked so far: 100


In [ ]:
info = pd.read_csv(f"{work}/selected_ligands.csv")
info = info[["molecule_chembl_id", "name", "smiles", "MW", "logP"]]

scores = scores.merge(info, on="molecule_chembl_id", how="left")
scores.to_csv(f"{work}/results/all_scores.csv", index=False)

top = scores.head(20)
top.to_csv(f"{work}/results/top20_scores.csv", index=False)

os.makedirs(top_dir, exist_ok=True)
for cid in top["molecule_chembl_id"]:
    shutil.copy(f"{res_dir}/{cid}_out.pdbqt", top_dir)
    shutil.copy(f"{work}/ligands/{cid}.pdbqt", top_dir)

shutil.copy(f"{work}/receptor_A.pdbqt", top_dir)
print(len(os.listdir(top_dir)), "files in top20 folder")
top[["molecule_chembl_id", "best_affinity_kcal_mol", "smiles"]]

41 files in top20 folder


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,molecule_chembl_id,best_affinity_kcal_mol,smiles
0,CHEMBL3394191,-11.355,C[C@H]1COC2(CCN(c3nc(=O)c4cc(C(F)(F)F)cc([N+](...
1,CHEMBL5193230,-10.997,O=c1ccc2cc(-n3cc(CSc4nc(=O)c5cc(Cl)cc([N+](=O)...
2,CHEMBL5563705,-10.938,C#Cc1c(C(F)(F)F)cc([N+](=O)[O-])c2sc(N3CCC4(CC...
3,CHEMBL5181669,-10.703,Cc1ccc([C@@]2(O)CCN(c3nc(=O)c4cc(C(F)(F)F)cc([...
4,CHEMBL4466845,-10.679,[2H]c1c(C(F)(F)F)c([2H])c2c(=O)nc(N3CCC4(CC3)O...
5,CHEMBL2177265,-10.527,O=c1nc(NCc2ccc3c(c2)OCCO3)sc2c([N+](=O)[O-])cc...
6,CHEMBL5171690,-10.486,O=c1nc(N2CC[C@](O)(COc3ccc(F)cc3)[C@@H](O)C2)s...
7,CHEMBL5191266,-10.365,O=c1nc(Nc2ccc3[nH]ncc3c2)sc2c([N+](=O)[O-])cc(...
8,CHEMBL5178291,-10.302,O=c1nc(N2CCc3[nH]nc(-c4ccccc4)c3C2)sc2c([N+](=...
9,CHEMBL5204154,-10.277,O=c1nc(N2CCn3nc(-c4ccccc4)nc3C2)sc2c([N+](=O)[...


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/docking_top20", "zip", f"{work}/results")
files.download("/content/docking_top20.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# --- Re-create the environment ---
!pip install -q rdkit meeko openbabel-wheel
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /content/vina
!chmod +x /content/vina

import os, numpy as np
from google.colab import files

work = "/content/docking_project"
os.makedirs(f"{work}/results", exist_ok=True)

# --- Upload the prepared receptor (Preparedsecond6hez.pdb) ---
uploaded = files.upload()
prep = "/content/" + list(uploaded.keys())[0]

# --- Download 6HEZ for the crystal ligand ---
os.system(f"wget -q https://files.rcsb.org/download/6HEZ.pdb -O {work}/6HEZ.pdb")

# --- Receptor: chain A + FAD, then convert to PDBQT ---
kept = []
for line in open(prep):
    rec_type = line[:6].strip()
    if rec_type == "ATOM" and line[21] == "A":
        kept.append(line)
    elif rec_type == "HETATM" and line[21] == "A" and line[17:20].strip() == "FAD":
        kept.append(line)
kept.append("END\n")
open(f"{work}/receptor_A.pdb", "w").writelines(kept)

rec = f"{work}/receptor_A.pdbqt"
!obabel {work}/receptor_A.pdb -O {rec} -xr -p 7.4

# --- Grid box from the chain A 0SK site ---
site = []
for l in open(f"{work}/6HEZ.pdb"):
    if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A":
        site.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
site = np.array(site)
c = site.mean(axis=0).round(2)
s = np.maximum(site.max(axis=0) - site.min(axis=0) + 10, 20).round(1)
print("receptor exists:", os.path.exists(rec))
print("centre:", c, "size:", s)

Saving Preparedsecond6hez.pdb to Preparedsecond6hez.pdb
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is /content/docking_project/receptor_A.pdb)

1 molecule converted
receptor exists: True
centre: [ 13.87 -20.62  37.13] size: [20.1 20.  20. ]


In [5]:
!pip install -q gemmi

In [7]:
import os, subprocess
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate

# 1. Extract the crystal ligand (0SK, chain A)
lines = [l for l in open(f"{work}/6HEZ.pdb")
         if l.startswith("HETATM") and l[17:20].strip() == "0SK" and l[21] == "A"]
open(f"{work}/0SK_crystal.pdb", "w").writelines(lines + ["END\n"])

# 2. Assign bond orders from the RCSB SMILES
template = Chem.MolFromSmiles("C[C@H]1COC2(CCN(CC2)C3=NC(=O)C4=CC(=CC(=C4S3)N=O)C(F)(F)F)O1")
crystal = Chem.MolFromPDBFile(f"{work}/0SK_crystal.pdb")
crystal = AllChem.AssignBondOrdersFromTemplate(template, crystal)

# 3. Prepare a docking input from the crystal coordinates
lig_h = Chem.AddHs(crystal, addCoords=True)
setup = MoleculePreparation().prepare(lig_h)[0]
pdbqt, ok, err = PDBQTWriterLegacy.write_string(setup)
open(f"{work}/0SK_redock_in.pdbqt", "w").write(pdbqt)

# 4. Dock with the same box and settings as the main run
out = f"{work}/results/0SK_redock_out.pdbqt"
cmd = ["/content/vina", "--receptor", rec, "--ligand", f"{work}/0SK_redock_in.pdbqt",
       "--center_x", str(c[0]), "--center_y", str(c[1]), "--center_z", str(c[2]),
       "--size_x", str(s[0]), "--size_y", str(s[1]), "--size_z", str(s[2]),
       "--exhaustiveness", "8", "--num_modes", "5", "--out", out]
subprocess.run(cmd, capture_output=True, text=True)

# 5. RMSD of each pose against the crystal pose (heavy atoms, no re-alignment)
pm = PDBQTMolecule.from_file(out, skip_typing=True)
docked = RDKitMolCreate.from_pdbqt_mol(pm)[0]
ref = Chem.RemoveHs(crystal)
for i in range(docked.GetNumConformers()):
    m = Chem.RemoveHs(Chem.Mol(docked, confId=i))
    print(f"pose {i+1}: RMSD = {rdMolAlign.CalcRMS(m, ref):.2f} Å")

[RDKit] WARNING:[08:00:13] WARNING: More than one matching pattern found - picking one
[RDKit] WARNING:


pose 1: RMSD = 9.43 Å
pose 2: RMSD = 7.47 Å
pose 3: RMSD = 7.36 Å
pose 4: RMSD = 8.16 Å
pose 5: RMSD = 2.37 Å


In [8]:
out2 = f"{work}/results/0SK_redock_ex32_out.pdbqt"
cmd = ["/content/vina", "--receptor", rec, "--ligand", f"{work}/0SK_redock_in.pdbqt",
       "--center_x", str(c[0]), "--center_y", str(c[1]), "--center_z", str(c[2]),
       "--size_x", str(s[0]), "--size_y", str(s[1]), "--size_z", str(s[2]),
       "--exhaustiveness", "32", "--num_modes", "20", "--energy_range", "4",
       "--seed", "42", "--out", out2]
subprocess.run(cmd, capture_output=True, text=True)

scores = [float(l.split()[3]) for l in open(out2) if l.startswith("REMARK VINA RESULT")]
pm = PDBQTMolecule.from_file(out2, skip_typing=True)
docked = RDKitMolCreate.from_pdbqt_mol(pm)[0]
ref = Chem.RemoveHs(crystal)
for i in range(docked.GetNumConformers()):
    m = Chem.RemoveHs(Chem.Mol(docked, confId=i))
    print(f"pose {i+1}: {scores[i]:.2f} kcal/mol, RMSD = {rdMolAlign.CalcRMS(m, ref):.2f} Å")

pose 1: -9.95 kcal/mol, RMSD = 9.45 Å
pose 2: -9.51 kcal/mol, RMSD = 7.48 Å
pose 3: -9.02 kcal/mol, RMSD = 7.34 Å
pose 4: -8.98 kcal/mol, RMSD = 8.16 Å
pose 5: -8.89 kcal/mol, RMSD = 7.52 Å
pose 6: -8.65 kcal/mol, RMSD = 2.40 Å
pose 7: -8.63 kcal/mol, RMSD = 2.28 Å
pose 8: -8.62 kcal/mol, RMSD = 2.21 Å
pose 9: -8.60 kcal/mol, RMSD = 3.97 Å
pose 10: -8.55 kcal/mol, RMSD = 7.48 Å
pose 11: -8.53 kcal/mol, RMSD = 8.22 Å
pose 12: -8.52 kcal/mol, RMSD = 1.60 Å
pose 13: -7.91 kcal/mol, RMSD = 2.42 Å
pose 14: -7.90 kcal/mol, RMSD = 9.29 Å
pose 15: -7.87 kcal/mol, RMSD = 6.68 Å
pose 16: -7.84 kcal/mol, RMSD = 8.55 Å
pose 17: -7.77 kcal/mol, RMSD = 3.58 Å
pose 18: -7.76 kcal/mol, RMSD = 8.63 Å
pose 19: -7.73 kcal/mol, RMSD = 9.10 Å
pose 20: -7.60 kcal/mol, RMSD = 3.99 Å


In [9]:
import os, shutil
from google.colab import files

out2 = f"{work}/results/0SK_redock_ex32_out.pdbqt"
pose_dir = f"{work}/results/0SK_poses"
os.makedirs(pose_dir, exist_ok=True)

# Split the multi-model PDBQT into one file per pose
poses, current = [], []
for line in open(out2):
    if line.startswith("MODEL"):
        current = [line]
    elif line.startswith("ENDMDL"):
        current.append(line)
        poses.append(current)
        current = []
    elif current:
        current.append(line)

for i, block in enumerate(poses, 1):
    with open(f"{pose_dir}/0SK_redock_pose{i:02d}.pdbqt", "w") as f:
        f.writelines(block)

# Include the crystal ligand for comparison in PyMOL
shutil.copy(f"{work}/0SK_crystal.pdb", f"{pose_dir}/0SK_crystal.pdb")

print(len(poses), "poses written:", sorted(os.listdir(pose_dir))[:3], "...")

# Zip and download
shutil.make_archive(f"{work}/results/0SK_poses", "zip", pose_dir)
files.download(f"{work}/results/0SK_poses.zip")

20 poses written: ['0SK_crystal.pdb', '0SK_redock_pose01.pdbqt', '0SK_redock_pose02.pdbqt'] ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>